# Run Greedy

Run this notebook for the selected `DATASET` / `MODEL` combo.

This notebook:
- Runs the official `GreedyCFExplainer` implementation from the vendored CoDy submodule.
- Loads the matching checkpoint and explain index for the chosen combo.
- Saves official run artifacts under `resources/results/official_greedy/` and finishes with the shared one-spot metrics summary.

Usage:
- Change `DATASET` and `MODEL` in the first code cell when needed.
- Run the notebook top to bottom.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_PROJECT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _PROJECT_CANDIDATES if (p / "I_explainer_benchmark").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from I_explainer_benchmark.src.notebooks.explainer_notebook_setup import (
    initialize_explainer_notebook,
    prepare_explainer_run,
)

# Select the dataset / model combo here.
DATASET = "simulate_v1"
MODEL = "tgn"

# Shared notebook setup. Leave the rest unchanged.
BOOT = initialize_explainer_notebook("03_greedy.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)
NB = BOOT.env
CFG = BOOT.config
SETTINGS = BOOT.settings
BENCHMARK_ROOT = BOOT.bench_root
REPO_ROOT = BOOT.repo_root
CODY_ROOT = BOOT.paths["CODY_ROOT"]
TEMGX_LINK = BOOT.paths["TEMGX_LINK"]


In [2]:
import torch
from I_explainer_benchmark.src.notebooks.notebook_helpers import set_global_seed

MODEL = str(MODEL).strip().lower()
SUPPORTED_MODEL_TYPES = set(CFG["supported_model_types"])
if MODEL not in SUPPORTED_MODEL_TYPES:
    raise ValueError(f"Unsupported MODEL={MODEL!r}. Choose one of {sorted(SUPPORTED_MODEL_TYPES)}")

CTX = prepare_explainer_run("03_greedy.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)

DATASET_NAME = CTX.dataset
MODEL_TYPE = CTX.model
SEED = int(CFG["seed"])
set_global_seed(SEED)

SELECTION_POLICY = str(SETTINGS["selection_policy"])
CANDIDATES_SIZE = int(SETTINGS["candidates_size"])
SAMPLE_SIZE = int(SETTINGS["sample_size"])
APPROXIMATE_PREDICTIONS = bool(SETTINGS["approximate_predictions"])
N_EVAL_EVENTS = int(SETTINGS["n_eval_events"])
EXPLAIN_START = int(SETTINGS.get("explain_start", 0))
PAPER_REDDIT_TGN_EVENT_PROTOCOL = bool(SETTINGS.get("paper_reddit_tgn_event_protocol", False))
PAPER_REDDIT_LOGIT_MIN = float(SETTINGS.get("paper_reddit_logit_min", 0.2))
PAPER_REDDIT_TRAIN_START = float(SETTINGS.get("paper_reddit_train_start", 0.2))
PAPER_REDDIT_TRAIN_END = float(SETTINGS.get("paper_reddit_train_end", 0.6))

FLIP_PROGRESS_BUDGET = float(CFG["flip_progress_budget"])
FLIP_PROGRESS_AREA_THRESHOLD = float(CFG["flip_progress_area_threshold"])
FLIP_PROGRESS_POINT_THRESHOLD = float(CFG["flip_progress_point_threshold"])
TGNN_LEVELS = list(CFG["tgnn_levels"])
RANKING_CFG = dict(CFG["ranking_cfg"])
SOURCE_NOTEBOOK = str(CFG["source_notebook"])

EXPLAIN_INDEX_PATH = CTX.explain_index_path
CKPT_PATH = CTX.checkpoint_path
DEVICE = CTX.device

print("DATASET:", DATASET_NAME)
print("MODEL  :", MODEL_TYPE)
print("CKPT   :", CKPT_PATH)
print("DEVICE :", DEVICE)


DATASET: simulate_v1
MODEL  : tgn
CKPT   : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/simulate_v1/tgn/tgn_simulate_v1_best.pth
DEVICE : mps


In [3]:
import inspect

from cody.explainer.greedy import GreedyCFExplainer

greedy_src = (inspect.getsourcefile(GreedyCFExplainer) or "").replace("\\", "/")
print("Greedy source:", greedy_src)
assert "/submodules/explainer/cody/cody/explainer/greedy.py" in greedy_src.lower(), (
    "Expected official GreedyCFExplainer from submodules/explainer/cody."
)


Greedy source: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/submodules/explainer/CoDy/cody/explainer/greedy.py


In [4]:
import numpy as np
import pandas as pd

from I_explainer_benchmark.src.data.io import load_processed_dataset
from I_explainer_benchmark.src.models.loader import load_backbone_model
from I_explainer_benchmark.src.notebooks.notebook_helpers import forced_target_event_ids_for_combo as _forced_target_event_ids_for_combo
from I_explainer_benchmark.src.notebooks.notebook_helpers import select_explain_event_ids as _select_explain_event_ids

bundle = load_processed_dataset(DATASET_NAME)
events = bundle["interactions"]
edge_features = bundle.get("edge_features")
node_features = bundle.get("node_features")

if edge_features is None or node_features is None:
    raise ValueError("CoDy requires edge and node features in the processed dataset bundle.")

def _is_bipartite_events(events_df: pd.DataFrame, dataset_name: str) -> bool:
    u_min, u_max = int(events_df["u"].min()), int(events_df["u"].max())
    i_min, i_max = int(events_df["i"].min()), int(events_df["i"].max())
    is_bip = bool(i_min > u_max or u_min > i_max)
    if str(dataset_name).lower() in {"stick_figure", "sticky_hips"}:
        is_bip = False
    return is_bip


model, _ = load_backbone_model(
    model_type=MODEL_TYPE,
    dataset_name=DATASET_NAME,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
)
backbone = model.backbone


FORCED_TARGET_EVENT_IDS = list(globals().get("FORCED_TARGET_EVENT_IDS", _forced_target_event_ids_for_combo(DATASET_NAME, MODEL_TYPE)))

all_event_ids, target_event_ids = _select_explain_event_ids(
    EXPLAIN_INDEX_PATH,
    n_eval_events=N_EVAL_EVENTS,
    start=EXPLAIN_START,
    include_event_ids=FORCED_TARGET_EVENT_IDS,
)

print("Loaded events rows:", len(events))
print("edge_features shape:", tuple(edge_features.shape))
print("node_features shape:", tuple(node_features.shape))
print("Target event ids:", target_event_ids)


102 events to explain
Loaded events rows: 16038
edge_features shape: (16039, 4)
node_features shape: (5, 4)
Target event ids: [3, 92, 142, 268, 386, 471, 711, 935, 1056, 1161, 1266, 1457, 1526, 1750, 1864, 1985, 2174, 2430, 2590, 2889, 3081, 3357, 3466, 3537, 3674, 3876, 4025, 4167, 4291, 4465]


In [5]:
from temgxlib.data import ContinuousTimeDynamicGraphDataset
from I_explainer_benchmark.src.models.adapters.wrapper import TGNNWrapper
from I_explainer_benchmark.src.notebooks.official_explainer_notebook_runtime import prepare_cody_runtime

_cody_prepared = prepare_cody_runtime(
    events=events,
    edge_features=edge_features,
    node_features=node_features,
    dataset_name=DATASET_NAME,
    backbone=backbone,
    device=DEVICE,
    dataset_cls=ContinuousTimeDynamicGraphDataset,
    tgnn_wrapper_cls=TGNNWrapper,
)

cody_events = _cody_prepared.cody_events
edge_features_for_cody = _cody_prepared.edge_features_for_cody
model_event_ids = _cody_prepared.model_event_ids
cody_dataset = _cody_prepared.cody_dataset
num_hops = _cody_prepared.num_hops
tgnn_wrapper = _cody_prepared.tgnn_wrapper
event_id_offset = _cody_prepared.event_id_offset

print("num_hops:", num_hops)
print("event_id_offset:", event_id_offset)


num_hops: 2
event_id_offset: 1


In [6]:
from cody.constants import COL_ID

from I_explainer_benchmark.src.notebooks.official_explainer_notebook_runtime import build_official_counterfactual_records

official_greedy = GreedyCFExplainer(
    tgnn_wrapper=tgnn_wrapper,
    selection_policy=SELECTION_POLICY,
    candidates_size=CANDIDATES_SIZE,
    sample_size=SAMPLE_SIZE,
    verbose=False,
    approximate_predictions=APPROXIMATE_PREDICTIONS,
)

_greedy_run = build_official_counterfactual_records(
    explainer=official_greedy,
    cody_events=cody_events,
    target_event_ids=target_event_ids,
    event_id_offset=event_id_offset,
    backbone=backbone,
    anchor_prefix='official_greedy',
    run_id='official_greedy',
    explainer_name='greedy',
    candidate_id_column=COL_ID,
    row_prefix_fields={'selection_policy': SELECTION_POLICY},
    row_suffix_fields={'sample_size': int(SAMPLE_SIZE)},
)

rows = _greedy_run.rows
cody_result_records = _greedy_run.result_records
results_df = _greedy_run.results_df
pass  # suppressed intermediate display; one-spot summary is shown below


In [7]:
# Shared metrics + export pipeline (clean notebook wrapper)
from I_explainer_benchmark.src.notebooks.notebook_metrics_pipeline import run_notebook_metrics_from_namespace

_pipeline_out = run_notebook_metrics_from_namespace(globals(), CFG)
globals().update(_pipeline_out)
run_dir_export = _pipeline_out['run_dir_export']
export_root = _pipeline_out['export_root']
out_jsonl = _pipeline_out['out_jsonl']
out_dir = _pipeline_out['out_dir']
base_name = _pipeline_out['base_name']
print('CURRENT_RUN_ID:', _pipeline_out['CURRENT_RUN_ID'])
print('Saved run export directory:', run_dir_export)
print('Updated global tables under:', export_root)


CURRENT_RUN_ID: simulate_v1_tgn_official_greedy_20260315_200831
Saved run export directory: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready/simulate_v1_tgn_official_greedy_20260315_200831
Updated global tables under: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready


In [8]:
# One-spot metrics summary (official)
from I_explainer_benchmark.src.notebooks.notebook_display import show_one_spot_metrics_summary

_one_spot = globals().get("one_spot", globals().get("_pipeline_out", {}).get("one_spot", {}))
show_one_spot_metrics_summary(_one_spot)


One-spot metrics summary (official):


,dataset,model,explainer,variant,n_events,best_fid,best_fid_sparsity,aufsc,best_minus_aufsc,best_fid_raw,...,tempme_acc_auc.ratio_acc,tempme_acc_auc.ratio_prob,tempme_acc_auc.ratio_logit,tempme_acc_auc.ratio_aps,tempme_acc_auc.ratio_auc,temgx_aufsc,cody_AUFSC_plus,cody_AUFSC_minus,cody_CHARR,elapsed_sec
0,simulate_v1,tgn,greedy,official,30,1.2122513379,1.0,-0.4676053372,1.6798566751,1.2122513379,...,0.7645833333,-0.0094804257,-0.1261249827,0.8322916667,0.9375,0.1170681119,0.7813207225,0.3991247413,0.528350425,2.2571107584
